# Aula Prática: Introdução à Resolução de Modelos Matemáticos com Python

**Objetivo da Aula:** Aprender a traduzir um modelo matemático conceitual para a linguagem de programação Python, utilizando bibliotecas de otimização (solvers) para encontrar a solução ótima. Neste momento, não nos preocuparemos em como o solver encontra a resposta, mas sim em como nos comunicamos com ele.

**Observação:** Criada iterativamente com o auxílio do Gemini. Ajustada posteriormente para adequações e complementações.

## Pré-requisitos (Instalação)

Antes de começar, certifique-se de ter os pacotes instalados no seu ambiente. Execute a célula abaixo caso não os tenha:

In [ ]:
!pip install gurobipy pyscipopt

In [2]:
!pip install pyscipopt

  Using cached pyscipopt-6.1.0-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.5 kB)
  Using cached numpy-2.4.4-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached pyscipopt-6.1.0-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (17.4 MB)
Using cached numpy-2.4.4-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyscipopt]/2 [pyscipopt]


**Importante:** 
- O Gurobi é um software comercial poderoso. A instalação via pip inclui uma licença restrita automática (tamanho limitado de variáveis e restrições), o que é suficiente para a aula, mas pode limitar a resolução da aplicação do projeto. O gurobi permiti que os alunos possam se cadastrar para obter uma licença de uso acadêmico no site da empresa. Basta comprovar o vínculo através de um e-mail institucional (@usp.br) e baixar a licença estando conectado na rede da universiddade (mapeamento de IP). Após instalar a licença é possível resolver os problemas, mesmo não estando dentro da rede da universidade. 
- O SCIP é um solver não-comercial. Por este motivo não demanda licença. 

## 1. O Problema: Planejamento de Produção (Mix de Produção)

Uma fábrica de brinquedos produz dois produtos: Carrinhos (A) e Bonecos (B). A fábrica deseja maximizar seu lucro diário, sabendo que:
- O lucro de cada Carrinho (A) é de R$ 40,00 e o do Boneco (B) é de R$ 50,00.
- Ambos passam por dois processos com recursos limitados:
    - Madeira: Temos apenas 40 kg disponíveis por dia. O produto A usa 1 kg e o produto B usa 2 kg.
    - Mão de obra: Temos apenas 60 horas disponíveis por dia. O produto A demanda 3 horas e o produto B demanda 2 horas.
- Não podemos produzir frações de brinquedos (as variáveis devem ser inteiras).

### Modelagem Matemática

Variáveis de Decisão:
- $x$: Quantidade de Carrinhos (A) a serem produzidos.
- $y$: Quantidade de Bonecos (B) a serem produzidos.

### Função Objetivo:
Maximizar $Z=40x+50y$

### Restrições:
$1x+2y≤40$ (Restrição de Madeira)

$3x+2y≤60$ (Restrição de Mão de Obra)

$x,y\in \mathcal{Z}^2_+$ (Domínio das variáveis)

## 2. Resolvendo com Gurobi (gurobipy)

O pacote gurobipy possui uma sintaxe muito orientada a objetos e é o padrão de mercado corporativo.

In [ ]:
import gurobipy as gp
from gurobipy import GRB

# 1. Inicializar o ambiente e o modelo
modelo_gurobi = gp.Model("Mix_Producao_Gurobi")

# 2. Instanciar Variáveis
# vtype=GRB.INTEGER garante que a resposta não será fracionada. 
# Para contínuas, usaríamos GRB.CONTINUOUS e para binárias GRB.BINARY
x = modelo_gurobi.addVar(vtype=GRB.INTEGER, name="Carrinhos_A", lb=0)
y = modelo_gurobi.addVar(vtype=GRB.INTEGER, name="Bonecos_B", lb=0)

# 3. Definir a Função Objetivo
# Queremos maximizar (GRB.MAXIMIZE) a equação de lucro
modelo_gurobi.setObjective(40 * x + 50 * y, GRB.MAXIMIZE)

# 4. Montar as Restrições
modelo_gurobi.addConstr(1 * x + 2 * y <= 40, name="Restricao_Madeira")
modelo_gurobi.addConstr(3 * x + 2 * y <= 60, name="Restricao_MaoDeObra")

# 5. Resolver o problema
modelo_gurobi.optimize()

# 6. Analisar os Resultados
print("\n--- RESULTADOS GUROBI ---")
if modelo_gurobi.status == GRB.OPTIMAL:
    print(f"Status: Solução Ótima Encontrada!")
    print(f"Lucro Máximo (Z): R$ {modelo_gurobi.objVal:.2f}")
    print(f"Produzir Carrinhos (x): {x.X}")
    print(f"Produzir Bonecos (y): {y.X}")
else:
    print("O solver não encontrou uma solução ótima. Verifique o modelo.")

## 3. Resolvendo com SCIP (pyscipopt)

O pacote pyscipopt é a interface Python para o solver SCIP. Você notará que a lógica é idêntica ao Gurobi, mudando apenas os nomes de algumas funções e parâmetros.

In [ ]:
from pyscipopt import Model

# 1. Inicializar o modelo
modelo_scip = Model("Mix_Producao_SCIP")

# Ocultar o log de processamento no terminal (opcional para manter o output limpo)
# modelo_scip.hideOutput(True)
# Redireciona o log interno do SCIP para o python e não terminal
modelo_scip.redirectOutput()

# 2. Instanciar Variáveis
# vtype="I" para Inteiras, "C" para Contínuas, "B" para Binárias.
x_scip = modelo_scip.addVar(vtype="I", name="Carrinhos_A", lb=0)
y_scip = modelo_scip.addVar(vtype="I", name="Bonecos_B", lb=0)

# 3. Definir a Função Objetivo e o sentido (maximize/minimize)
modelo_scip.setObjective(40 * x_scip + 50 * y_scip, sense="maximize")

# 4. Montar as Restrições (usamos addCons no SCIP)
modelo_scip.addCons(1 * x_scip + 2 * y_scip <= 40, name="Restricao_Madeira")
modelo_scip.addCons(3 * x_scip + 2 * y_scip <= 60, name="Restricao_MaoDeObra")
modelo_scip.addCons(x_scip <= 15, name="Restricao_Mercado")


# 5. Resolver o problema
modelo_scip.optimize()


# 6. Analisar os Resultados
print("\n--- RESULTADOS SCIP ---")
if modelo_scip.getStatus() == "optimal":
    print(f"Status: Solução Ótima Encontrada!")
    # getObjVal() pega o valor da função objetivo
    print(f"Lucro Máximo (Z): R$ {modelo_scip.getObjVal():.2f}")
    # getVal() pega o valor de uma variável específica
    print(f"Produzir Carrinhos (x): {modelo_scip.getVal(x_scip)}")
    print(f"Produzir Bonecos (y): {modelo_scip.getVal(y_scip)}")
else:
    print("O solver não encontrou uma solução ótima. Verifique o modelo.")

## 4. Exercício de Fixação para os Alunos

Para praticar agora:

Modifique o código acima (escolha o seu solver favorito) para refletir a seguinte mudança de cenário:
1. O mercado só consegue absorver no máximo 15 carrinhos por dia. Adicione esta nova restrição.
2. Resolva novamente. Como isso impactou o lucro máximo e a produção de bonecos?

## 5. Dimensionamento de Lotes com Múltiplos Itens com Restrições de Capacidade

Quando o número de itens cresce, digitar listas e dicionários na mão se torna inviável. Em ambientes reais, importaríamos esses dados de um arquivo Excel, CSV ou banco de dados. Para a nossa aula, vamos gerar esses parâmetros usando laços de repetição e manter a modelagem flexível.

In [ ]:
from pyscipopt import Model
import random

# Definir semente para garantir que todos na aula tenham os mesmos dados gerados
random.seed(133)

# ---------------------------------------------------------
# 1. Dados de Entrada (Gerados Dinamicamente)
# ---------------------------------------------------------
itens = [f"P{i}" for i in range(1, 12)]  # Conjunto de Itens
meses = [i for i in range(1,10)]          # Horizonte de planejamento (conjunto de meses)

# Gerando parâmetros aleatórios realistas
custo_producao = {i: random.randint(10, 20) for i in itens}
custo_setup = {i: random.randint(300, 500) for i in itens}
custo_estoque = {i: random.randint(1, 4) for i in itens}
tempo_maquina = {i: random.randint(1, 3) for i in itens}

# Demanda e Capacidade
demanda = {(i, t): random.randint(0, 5)*10 for i in itens for t in meses}
capacidade_mensal = {t: 50*len(itens) for t in meses} # horas disponíveis por mês (proporcional a quantidade de itens)

# ---------------------------------------------------------
# 2. Inicialização do Modelo SCIP
# ---------------------------------------------------------
modelo_scip = Model("Dimensionamento_Lotes_8_Itens")
# Redireciona o log interno do SCIP para o python e não terminal
modelo_scip.redirectOutput()

# ---------------------------------------------------------
# 3. Instanciando Variáveis
# ---------------------------------------------------------
x = {} # Produção
y = {} # Setup
e = {} # Estoque

for i in itens:
    for t in meses:
        x[i, t] = modelo_scip.addVar(vtype="I", name=f"Producao_{i}_{t}", lb=0)
        y[i, t] = modelo_scip.addVar(vtype="B", name=f"Setup_{i}_{t}")
        e[i, t] = modelo_scip.addVar(vtype="I", name=f"Estoque_{i}_{t}", lb=0)

# ---------------------------------------------------------
# 4. Função Objetivo
# No SCIP, a função embutida sum() do Python funciona perfeitamente
# ---------------------------------------------------------
objetivo = sum(custo_producao[i] * x[i, t] + custo_setup[i] * y[i, t] 
               + custo_estoque[i] * e[i, t] for i in itens for t in meses)

modelo_scip.setObjective(objetivo, sense="minimize")

# ---------------------------------------------------------
# 5. Famílias de Restrições
# ---------------------------------------------------------

# 5.1 Balanço de Estoque
for i in itens:
    modelo_scip.addCons(x[i, meses[0]] == demanda[i, meses[0]] + e[i, meses[0]], 
    name=f"Balanco_{i}_{meses[0]}")
    for t in meses[1:]:
        modelo_scip.addCons(e[i, t-1] + x[i, t] == demanda[i, t] + e[i, t], 
            name=f"Balanco_{i}_{t}")

# 5.2 Lógica de Setup (Big-M)
for i in itens:
    for t in meses:
        M_dinamico = sum(demanda[i, tau] for tau in meses if tau >= t)
        modelo_scip.addCons(x[i, t] <= M_dinamico * y[i, t], name=f"SetupLogic_{i}_{t}")

# 5.3 Capacidade
for t in meses:
    uso_capacidade = sum(tempo_maquina[i] * x[i, t] for i in itens)
    modelo_scip.addCons(uso_capacidade <= capacidade_mensal[t], 
        name=f"Capacidade_{t}")

# ---------------------------------------------------------
# 6. Otimização e Análise
# ---------------------------------------------------------
modelo_scip.optimize()

print("\n" + "="*50)
print("PLANO DE PRODUÇÃO ÓTIMO (SCIP - 8 ITENS)")
print("="*50)

if modelo_scip.getStatus() == "optimal":
    print(f"Custo Total Minimizado: R$ {modelo_scip.getObjVal():.2f}\n")
    
    for t in meses:
        print(f"--- MÊS {t} ---")
        for i in itens:
            # Pegando os valores das variáveis após a otimização
            prod = modelo_scip.getVal(x[i, t])
            estq = modelo_scip.getVal(e[i, t])
            setup = modelo_scip.getVal(y[i, t])
            
            # Formatação de string para deixar o relatório alinhado
            print(f"Item {i} -> Dem: {int(demanda[i,t]):2d} | Prod: {int(prod):2d} un | Setup: {'Sim' if setup > 0.5 else 'Não':3s} | Estoque: {int(estq):2d} un")
        print("-" * 25)
else:
    print("O solver não encontrou uma solução ótima.")